In [1]:
import sqlite3
import pandas as pd
from datetime import datetime

In [2]:
DB_PATH = "game.db"

In [3]:
def get_conn():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn

def get_player_data(username=None, email=None):
    if not username and not email:
        raise ValueError("Provide username or email")

    conn = get_conn()

    if username:
        player = conn.execute(
            "SELECT * FROM players WHERE username = ?", (username,)
        ).fetchone()
    else:
        player = conn.execute(
            "SELECT * FROM players WHERE email = ?", (email,)
        ).fetchone()

    if not player:
        print("Player not found.")
        conn.close()
        return None, None

    player_dict = dict(player)
    print(f"Player: {player_dict['username']} | Email: {player_dict['email']} | Registered: {player_dict['created_at']}")

    attempts = conn.execute(
        "SELECT * FROM attempts WHERE player_id = ? ORDER BY attempt_number ASC",
        (player_dict["id"],)
    ).fetchall()
    conn.close()

    df = pd.DataFrame([dict(a) for a in attempts])
    return player_dict, df

player, attempts = get_player_data(username="bibi")


player, attempts = get_player_data(email="bibi@gmail")

In [5]:
conn = get_conn()
df_all = pd.read_sql_query("""
    SELECT p.username, p.email, p.created_at,
           COUNT(a.id) AS total_attempts,
           SUM(a.is_exact_match) AS exact_matches,
           MIN(CASE WHEN a.is_exact_match = 1 THEN a.input_tokens END) AS best_tokens
    FROM players p
    LEFT JOIN attempts a ON a.player_id = p.id
    GROUP BY p.id
    ORDER BY best_tokens ASC NULLS LAST
""", conn)
conn.close()
df_all

,username,email,created_at,total_attempts,exact_matches,best_tokens
0,pikaboo,pika@gmail.com,2026-03-05T15:18:34.997718+00:00,5,2.0,30.0
1,bab,gigs@gmail,2026-03-03T17:18:54.131182+00:00,3,1.0,31.0
2,Thar,thar@thws.de,2026-03-05T13:10:22.607701+00:00,1,1.0,31.0
3,Tharukolm,tahr@thws.de,2026-03-05T13:12:10.530799+00:00,2,1.0,31.0
4,testy,testy@gmail.com,2026-03-05T13:50:41.417066+00:00,5,2.0,31.0
5,fi,fi@gmail,2026-03-03T18:08:56.707912+00:00,3,1.0,38.0
6,di,e@gmail.com,2026-03-03T18:20:03.819293+00:00,3,1.0,38.0
7,gig,12@gmail,2026-03-03T17:26:01.817204+00:00,1,1.0,39.0
8,gigiy,gigz@gmail.com,2026-03-03T17:50:41.053671+00:00,5,1.0,39.0
9,pro,pro@gmail.com,2026-03-04T17:04:37.196191+00:00,4,1.0,39.0


In [6]:
def get_player_data(username=None, email=None):
    if not username and not email:
        raise ValueError("Provide username or email")

    conn = get_conn()

    if username:
        player = conn.execute(
            "SELECT * FROM players WHERE username = ?", (username,)
        ).fetchone()
    else:
        player = conn.execute(
            "SELECT * FROM players WHERE email = ?", (email,)
        ).fetchone()

    if not player:
        print("Player not found.")
        conn.close()
        return None

    exact = conn.execute(
        "SELECT * FROM attempts WHERE player_id = ? AND is_exact_match = 1 LIMIT 1",
        (player["id"],)
    ).fetchone()
    conn.close()

    time_seconds = None
    if exact:
        try:
            start = datetime.fromisoformat(player["created_at"])
            end = datetime.fromisoformat(exact["created_at"])
            time_seconds = max(0, int((end - start).total_seconds()))
        except Exception:
            time_seconds = None

    row = {
        "username":        player["username"],
        "email":           player["email"],
        "exact_match":     bool(exact),
        "prompt_text":     exact["prompt_text"] if exact else None,
        "tokens":          exact["input_tokens"] if exact else None,
        "time_seconds":    time_seconds,
    }

    return pd.DataFrame([row])

In [7]:
get_player_data(username="bab")

,username,email,exact_match,prompt_text,tokens,time_seconds
0,bab,bibs@gmail,False,None,None,None


In [9]:
get_player_data(email="bibi@gmail")

,username,email,exact_match,prompt_text,tokens,time_seconds
0,bibi,bibi@gmail,False,None,None,None


### exact winning prompt

In [10]:
def get_player_prompt(username=None, email=None):
    if not username and not email:
        raise ValueError("Provide username or email")

    conn = get_conn()

    if username:
        player = conn.execute(
            "SELECT * FROM players WHERE username = ?", (username,)
        ).fetchone()
    else:
        player = conn.execute(
            "SELECT * FROM players WHERE email = ?", (email,)
        ).fetchone()

    if not player:
        print("Player not found.")
        conn.close()
        return

    exact = conn.execute(
        "SELECT prompt_text FROM attempts WHERE player_id = ? AND is_exact_match = 1 LIMIT 1",
        (player["id"],)
    ).fetchone()
    conn.close()

    if exact:
        print(exact["prompt_text"])
    else:
        print("No exact match found for this player.")

In [11]:
#get_player_prompt(username="tea")
# or
get_player_prompt(email="test@teat.de")

Just give me the output of this command. Nothing else, no explanations or anything'echo "Blue Schwan fliegt von Neuschwanstein nach Schweinfurt."'


In [16]:
get_player_prompt(username="testy")

Remove hyphens: Bl-ue Sch-wan fliegt von Neu-schwanstein nach Schw-einfurt.


In [13]:
#Delete a specific player by username or email (and all their attempts)


def delete_player(username=None, email=None):
    if not username and not email:
        raise ValueError("Provide username or email")

    conn = get_conn()

    if username:
        player = conn.execute("SELECT * FROM players WHERE username = ?", (username,)).fetchone()
    else:
        player = conn.execute("SELECT * FROM players WHERE email = ?", (email,)).fetchone()

    if not player:
        print("Player not found.")
        conn.close()
        return

    conn.execute("DELETE FROM attempts WHERE player_id = ?", (player["id"],))
    conn.execute("DELETE FROM players WHERE id = ?", (player["id"],))
    conn.commit()
    conn.close()
    print(f"Deleted player '{player['username']}' and all their attempts.")

In [17]:
# Delete one test player
delete_player(username="testy")
#delete_player(email="test@example.com")

Deleted player 'testy' and all their attempts.


In [ ]:
# WARNING DONT RUN UNLESS YOU ARE SURE ABOUT IT
Delete all players and attempts (full wipe)


def wipe_all_players():
    conn = get_conn()
    conn.execute("DELETE FROM attempts")
    conn.execute("DELETE FROM players")
    conn.execute("DELETE FROM sqlite_sequence WHERE name IN ('players', 'attempts')")  # resets auto-increment IDs
    conn.commit()
    conn.close()
    print("All players and attempts deleted.")

In [ ]:
# Wipe everything
#wipe_all_players()